# Colab Worker — Corpo e Mente

**Modelos cargados:**
- `Qwen3-VL-Embedding-2B` — embeddings multimodales de texto e imágenes (2048 dims)
- `YOLO26x-pose` — 17 keypoints COCO (x, y) por persona
- `YOLO26x-depth` — profundidad métrica Z en metros reales

**Fusión 3D (RF-03):** cada keypoint se enriquece con `z = depth_map[y_d, x_d]`
en metros absolutos, produciendo coordenadas `(x, y, z)` en $\mathbb{R}^3$.

**Runtime recomendado:** GPU G4 (preferido) · A100/H100 (alternativas potentes) · L4/T4 solo para pruebas cortas

In [ ]:
!pip install -q sentence-transformers langchain-text-splitters opencv-python-headless pillow numpy ultralytics psutil google-api-python-client google-auth-httplib2 google-auth


In [ ]:
# Celda 2: Diagnóstico de hardware
import os
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("colab_worker")

import cv2
import time
import subprocess
import sys
import numpy as np
import requests
import psutil
import torch
from PIL import Image

# --- Diagnóstico de GPU y RAM ---
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🖥️  GPU: {gpu_name}")
    print(f"💾 VRAM total: {vram_total_gb:.2f} GB")
else:
    print("⚠️  No se detectó GPU. Los modelos correrán en CPU (muy lento).")

ram_total_gb = psutil.virtual_memory().total / 1e9
print(f"🧠 RAM total: {ram_total_gb:.2f} GB")

# Verificar dependencias
import ultralytics
from sentence_transformers import SentenceTransformer
print(f"✅ ultralytics versión: {ultralytics.__version__}")
print("✅ Dependencias verificadas")


In [ ]:
# Celda 3: Carga de los 3 modelos en VRAM
import torch
import logging
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ultralytics import YOLO

logger = logging.getLogger("colab_worker")

# --- 1. Estado inicial de VRAM ---
if torch.cuda.is_available():
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# --- 2. Cargar Qwen ---
print("\nCargando Qwen3-VL-Embedding-2B...")
qwen_model = SentenceTransformer('Qwen/Qwen3-VL-Embedding-2B')
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
print("✅ Qwen cargado. Dim:", qwen_model.get_sentence_embedding_dimension())

if torch.cuda.is_available():
    print(f"VRAM usada tras Qwen: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM reservada tras Qwen: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# --- 3. Cargar los detectores con fallback ante falta de VRAM ---
def _load_yolo_model(primary_name, fallback_name, label):
    for model_name in (primary_name, fallback_name):
        try:
            model = YOLO(model_name)
            if torch.cuda.is_available():
                try:
                    model.to("cuda")
                    logger.info("%s cargado en GPU: %s", label, model_name)
                    return model, model_name, 0
                except Exception as gpu_error:
                    logger.warning("No se pudo cargar %s en GPU (%s).", model_name, gpu_error)
                    del model
                    import gc
                    gc.collect()
                    torch.cuda.empty_cache()
                    if model_name == fallback_name:
                        model = YOLO(model_name)
                        logger.warning("Se usará %s en CPU por límite de VRAM.", label)
                        return model, model_name, "cpu"
                    continue
            logger.warning("No hay GPU; %s se ejecutará en CPU.", label)
            return model, model_name, "cpu"
        except Exception as model_error:
            logger.warning("No se pudo cargar %s (%s).", model_name, model_error)
            if model_name == fallback_name:
                raise
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    raise RuntimeError(f"No se pudo cargar el modelo {label}.")


print("\nCargando YOLO pose...")
yolo_pose, pose_model_name, pose_device = _load_yolo_model(
    "yolo26x-pose.pt", "yolo26n-pose.pt", "pose"
)
print(f"✅ Pose cargado: {pose_model_name} ({'GPU' if pose_device == 0 else 'CPU'})")

print("\nCargando YOLO depth...")
yolo_depth, depth_model_name, depth_device = _load_yolo_model(
    "yolo26x-depth.pt", "yolo26n-depth.pt", "depth"
)
print(f"✅ Depth cargado: {depth_model_name} ({'GPU' if depth_device == 0 else 'CPU'})")

# --- 5. Verificar coexistencia ---
print("\nVerificando que Qwen sigue operativo...")
test_emb = qwen_model.encode(["prueba de coexistencia"], normalize_embeddings=True)
print(f"✅ Qwen responde. Dim del vector: {len(test_emb[0])}")

print("\n✅ Coexistencia triple verificada: Qwen + YOLO26-pose + YOLO26-depth")

In [ ]:
# Celda 4: Worker Asíncrono de Inferencia con Google Drive DataLake (Cero Túneles)
import os
import io
import time
import json
import logging
import cv2
import torch
import numpy as np
from itertools import zip_longest
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
logger = logging.getLogger("colab_worker_datalake")

# 1. Configuración del DataLake en Google Drive
DRIVE_FOLDER_ID = "1bxH5ZuvL-yxGqAZd5pbd7pLRoSI8gSwK"
TOKEN_FILE = "token.json"

if not os.path.exists(TOKEN_FILE):
    raise FileNotFoundError(f"❌ Sube tu archivo '{TOKEN_FILE}' al panel izquierdo de archivos de Colab para continuar.")

# 2. Inicializar cliente oficial de Google Drive
scopes = ["https://www.googleapis.com/auth/drive"]
creds = Credentials.from_authorized_user_file(TOKEN_FILE, scopes=scopes)
drive_service = build("drive", "v3", credentials=creds)
logger.info("✅ Conexión exitosa a Google Drive DataLake (Carpeta: %s)", DRIVE_FOLDER_ID)


def descargar_archivo_drive(file_id, save_path):
    """Descarga un archivo binario desde Google Drive."""
    try:
        request = drive_service.files().get_media(fileId=file_id)
        with io.FileIO(save_path, "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
        return True
    except Exception as e:
        logger.error("Error descargando archivo %s de Drive: %s", file_id, e)
        return False


def subir_json_resultado(nombre_archivo, datos):
    """Sube un archivo JSON con los resultados a la carpeta del DataLake en Drive."""
    try:
        metadata = {
            "name": nombre_archivo,
            "parents": [DRIVE_FOLDER_ID]
        }
        json_bytes = json.dumps(datos, ensure_ascii=False, indent=2).encode("utf-8")
        media = MediaIoBaseUpload(io.BytesIO(json_bytes), mimetype="application/json")
        created = drive_service.files().create(body=metadata, media_body=media, fields="id, name").execute()
        logger.info("✓ Resultado subido a Drive: %s (ID: %s)", nombre_archivo, created.get("id"))
        return created.get("id")
    except Exception as e:
        logger.error("Error subiendo resultado %s a Drive: %s", nombre_archivo, e)
        return None


def eliminar_archivo_drive(file_id):
    """Elimina un archivo procesado de Drive para mantener limpio el DataLake."""
    try:
        drive_service.files().delete(fileId=file_id).execute()
        logger.info("✓ Tarea eliminada de Drive: %s", file_id)
        return True
    except Exception as e:
        logger.warning("No se pudo eliminar archivo %s: %s", file_id, e)
        return False


def procesar_evaluacion_biomecanica(local_path, persona_idx=0):
    """
    Ejecuta la extracción cinemática de keypoints YOLOv26 Pose y Depth Fusion 3D
    (Pipeline completo de la Iteración C12.3 / C12.7).
    """
    logger.info("→ Iniciando inferencia cinemática YOLOv26 Pose + Depth 3D...")
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    global yolo_pose, yolo_depth, pose_device, depth_device
    p_dev = pose_device if "pose_device" in globals() else 0
    d_dev = depth_device if "depth_device" in globals() else 0

    pose_results = yolo_pose.predict(
        source=local_path, device=p_dev, stream=True, verbose=False, conf=0.3
    )
    depth_results = yolo_depth.predict(
        source=local_path, device=d_dev, stream=True, verbose=False, imgsz=768
    )

    cap = cv2.VideoCapture(local_path)
    orig_w = float(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 640)
    orig_h = float(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 640)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    estimated_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()

    if fps > 0 and estimated_frames > 0:
        logger.info("Video duración estimada %.1f s; fps=%.2f", estimated_frames / fps, fps)

    esqueletos, frames_con_persona = [], 0
    persona_anterior, frames_sin_deteccion = None, 0
    _SENTINEL = object()

    for frame_idx, pair in enumerate(zip_longest(pose_results, depth_results, fillvalue=_SENTINEL)):
        r_pose, r_depth = pair
        if r_pose is _SENTINEL or r_depth is _SENTINEL:
            break
        try:
            if r_pose.keypoints is None or len(r_pose.keypoints.data) == 0:
                esqueletos.append({"frame_idx": frame_idx, "keypoints133": [0.0] * 133})
                frames_sin_deteccion += 1
                if frames_sin_deteccion > 5:
                    persona_anterior = None
                continue

            raw = r_pose.keypoints.data.cpu().numpy()
            xy = r_pose.keypoints.xy.cpu().numpy()
            candidates = [i for i, person in enumerate(raw) if person.shape[0] >= 17]
            if not candidates:
                esqueletos.append({"frame_idx": frame_idx, "keypoints133": [0.0] * 133})
                frames_sin_deteccion += 1
                continue

            if persona_anterior is None:
                selected_idx = candidates[min(max(persona_idx, 0), len(candidates) - 1)]
            else:
                distances = []
                for candidate_idx in candidates:
                    candidate = xy[candidate_idx]
                    mask = (candidate[:, 0] > 0) & (candidate[:, 1] > 0)
                    if int(mask.sum()) < 5:
                        distances.append((float("inf"), candidate_idx))
                        continue
                    center = candidate[mask].mean(axis=0) / np.array([orig_w, orig_h])
                    previous = persona_anterior / np.array([orig_w, orig_h])
                    distances.append((float(np.linalg.norm(center - previous)), candidate_idx))
                distance, selected_idx = min(distances)
                if distance > 0.25:
                    esqueletos.append({"frame_idx": frame_idx, "keypoints133": [0.0] * 133})
                    frames_sin_deteccion += 1
                    if frames_sin_deteccion > 5:
                        persona_anterior = None
                    continue

            person_xy = xy[selected_idx]
            mask = (person_xy[:, 0] > 0) & (person_xy[:, 1] > 0)
            if int(mask.sum()) < 5:
                esqueletos.append({"frame_idx": frame_idx, "keypoints133": [0.0] * 133})
                frames_sin_deteccion += 1
                continue
            persona_anterior = person_xy[mask].mean(axis=0)

            depth_map = r_depth.depth.data.cpu().numpy()
            if depth_map.ndim == 3:
                depth_map = depth_map[0]
            h_depth, w_depth = depth_map.shape[-2:]
            sx, sy = w_depth / orig_w, h_depth / orig_h
            kpts_3d = []
            for x, y in person_xy[:17]:
                x_d = int(np.clip(x * sx, 0, w_depth - 1))
                y_d = int(np.clip(y * sy, 0, h_depth - 1))
                kpts_3d.extend([float(x), float(y), float(depth_map[y_d, x_d])])
            kpts_3d.extend([0.0] * (133 - len(kpts_3d)))
            esqueletos.append({"frame_idx": frame_idx, "keypoints133": kpts_3d[:133]})
            frames_con_persona += 1
            frames_sin_deteccion = 0
        finally:
            del r_pose, r_depth

    logger.info("Pose completada; total_frames=%s, con_persona=%s", len(esqueletos), frames_con_persona)
    return esqueletos


def ejecutar_bucle_worker():
    """Sondea continuamente Google Drive buscando tareas pendientes generadas por el backend."""
    logger.info("🚀 Worker Corpo e Mente iniciado en modo DataLake Google Drive.")
    logger.info("👂 Escuchando tareas en carpeta: %s...", DRIVE_FOLDER_ID)
    
    while True:
        try:
            query = f"'{DRIVE_FOLDER_ID}' in parents and name contains 'tarea_' and trashed = false"
            res = drive_service.files().list(q=query, fields="files(id, name)").execute()
            archivos_tarea = res.get("files", [])
            
            for t_file in archivos_tarea:
                t_id = t_file["id"]
                t_name = t_file["name"]
                logger.info("📌 Nueva tarea detectada en Drive: %s (ID: %s)", t_name, t_id)
                
                local_tarea_path = f"/tmp/{t_name}"
                if not descargar_archivo_drive(t_id, local_tarea_path):
                    continue
                
                with open(local_tarea_path, "r", encoding="utf-8") as f:
                    tarea_data = json.load(f)
                
                tipo = tarea_data.get("tipo", "video_evaluacion")
                
                if tipo == "video_evaluacion":
                    eval_id = tarea_data.get("evaluacion_id")
                    video_drive_id = tarea_data.get("video_drive_id")
                    persona_idx = tarea_data.get("persona_idx", 0)
                    
                    local_video = f"/tmp/video_{eval_id}.mp4"
                    logger.info("→ Descargando video de evaluación %s desde Drive...", eval_id)
                    if descargar_archivo_drive(video_drive_id, local_video):
                        # Procesar con YOLO Pose + Depth 3D
                        esqueletos = procesar_evaluacion_biomecanica(local_video, persona_idx)
                        
                        tot_f = len(esqueletos)
                        frames_ok = sum(1 for item in esqueletos if any(v > 0 for v in item.get("keypoints133", [])[:51]))
                        cob = (frames_ok / tot_f) if tot_f > 0 else 0.0
                        resultado_payload = {
                            "evaluacion_id": eval_id,
                            "tipo": "video_evaluacion",
                            "estado": "completado",
                            "esqueletos": esqueletos,
                            "total_esqueletos": tot_f,
                            "total_frames_procesados": tot_f,
                            "frames_con_persona": frames_ok,
                            "cobertura": cob
                        }
                        
                        # Subir resultado a Drive
                        subir_json_resultado(f"resultado_video_{eval_id}.json", resultado_payload)
                        
                        # Limpiar video local y tarea de Drive
                        if os.path.exists(local_video):
                            os.remove(local_video)
                        eliminar_archivo_drive(t_id)

                elif tipo == "qwen_embed_texto":
                    job_id = tarea_data.get("job_id")
                    texto = tarea_data.get("texto", "")
                    logger.info("→ Vectorizando texto con Qwen3-VL (job_id: %s)...", job_id)
                    chunks = splitter.split_text(texto) if "splitter" in globals() else [texto]
                    if chunks:
                        embs = qwen_model.encode(chunks, batch_size=16, normalize_embeddings=True)
                        embs_list = embs.tolist()
                    else:
                        embs_list = []
                    res_payload = {
                        "job_id": job_id,
                        "tipo": "qwen_embed_texto",
                        "estado": "completado",
                        "chunks": chunks,
                        "embeddings": embs_list,
                        "dimension": 2048
                    }
                    subir_json_resultado(f"resultado_qwen_texto_{job_id}.json", res_payload)
                    eliminar_archivo_drive(t_id)

                elif tipo == "qwen_embed_imagenes":
                    job_id = tarea_data.get("job_id")
                    imagenes_info = tarea_data.get("imagenes", [])
                    logger.info("→ Vectorizando %s imágenes con Qwen3-VL (job_id: %s)...", len(imagenes_info), job_id)
                    import base64
                    from PIL import Image
                    chunks = []
                    embs_list = []
                    for img_info in imagenes_info:
                        try:
                            img_bytes = base64.b64decode(img_info["b64"])
                            image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
                            emb = qwen_model.encode([image], normalize_embeddings=True)
                            chunks.append(img_info.get("label", "[Imagen]"))
                            embs_list.append(emb[0].tolist())
                        except Exception as img_err:
                            logger.error("Error vectorizando imagen: %s", img_err)
                    res_payload = {
                        "job_id": job_id,
                        "tipo": "qwen_embed_imagenes",
                        "estado": "completado",
                        "chunks": chunks,
                        "embeddings": embs_list,
                        "dimension": 2048
                    }
                    subir_json_resultado(f"resultado_qwen_img_{job_id}.json", res_payload)
                    eliminar_archivo_drive(t_id)
                
                if os.path.exists(local_tarea_path):
                    os.remove(local_tarea_path)
                    
        except Exception as e:
            logger.error("Error en el bucle del Worker: %s", e)
            
        time.sleep(5)


# Iniciar el proceso de escucha asíncrono puro en Drive
ejecutar_bucle_worker()


## Apagar el worker

Cuando termines las pruebas de inferencia para la tesis, usa **Entorno de ejecución → Desconectar y eliminar entorno de ejecución** en Colab para liberar la GPU asignada.
